# Synthetic experiments (unconfounded)

This notebook reproduces the synthetic experiments for the **unconfounded** setting.

All the experiments in this notebook are controlled by a **single setting**: the factual event indicator `E_FACTUAL` (0 or 1) in the config cell below.

- Set `E_FACTUAL = 1` to treat the **event** series as factual and the **no-event** series as counterfactual (default).
- Set `E_FACTUAL = 0` to treat the **no-event** series as factual and the **event** series as counterfactual.



In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

from models import CEPAE, CVAE, CAAE
from models.baselines import ForecastModel, EventPredictor
from models.caae_trainer import train as train_caae
from models.adversarial_forecast_model import AdversarialForecastModel, Trainer

from metrics.cf_metrics import (
    counterfactual_mae_mbe,
    axiomatic_metrics,
    added_variations_relative,
)

from data.synthetic_data_generators import (
    create_dataset,
    create_dataset_counterfactuals,
    create_dataset_confounded,
    create_dataset_counterfactuals_confounded,
)


## Data generation

In [ ]:
# Reproducibility
SEED = 123
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Synthetic series settings
SEQ_LEN = 30
KEY_STEP = 20  # split point between history (x) and future (y)
UNIFORM_CHANGE = 0.7
NOISE_STD = 0.1

# Factual/counterfactual event flags
E_FACTUAL = 1
E_COUNTERFACTUAL = 1 - E_FACTUAL

N_TRAIN = 2000
N_EVAL = 500
N_TEST = 500

HORIZON = SEQ_LEN - KEY_STEP  # predicted steps

# Generate observed train/eval data
train_labels, train_data = create_dataset(
    n=N_TRAIN, seq_len=SEQ_LEN, key_step=KEY_STEP,
    uniform_change=UNIFORM_CHANGE, scale_param=NOISE_STD, seed=SEED
)
eval_labels, eval_data = create_dataset(
    n=N_EVAL, seq_len=SEQ_LEN, key_step=KEY_STEP,
    uniform_change=UNIFORM_CHANGE, scale_param=NOISE_STD, seed=SEED + 1
)
y0_test, y1_test = create_dataset_counterfactuals(
    n=N_TEST, seq_len=SEQ_LEN, key_step=KEY_STEP,
    uniform_change=UNIFORM_CHANGE, scale_param=NOISE_STD, seed=SEED + 2
)

x_train, y_train = train_data[:, :KEY_STEP, :], train_data[:, KEY_STEP:, :]
x_eval, y_eval = eval_data[:, :KEY_STEP, :], eval_data[:, KEY_STEP:, :]

x_test = y0_test[:, :KEY_STEP, :]
y0_test = y0_test[:, KEY_STEP:, :]
y1_test = y1_test[:, KEY_STEP:, :]

# Labels for counterfactual generation
label_real_test = np.full((len(x_test), 1), float(E_FACTUAL), dtype=np.float32)
label_cf_test = np.full((len(x_test), 1), float(E_COUNTERFACTUAL), dtype=np.float32)

# Select factual and counterfactual potential outcomes automatically
y_factual_test = y1_test if E_FACTUAL == 1 else y0_test
y_counterfactual_test = y1_test if E_COUNTERFACTUAL == 1 else y0_test


## Event predictor (effectiveness metric)

In [ ]:
predictor = EventPredictor()
predictor.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"],
)

predictor.fit(
    y_train,
    train_labels,
    epochs=200,
    batch_size=32,
    validation_data=(y_eval, eval_labels),
    verbose=0,
)


## Evaluation helpers

In [ ]:
def evaluate_counterfactuals(model, name: str):
    pred = np.asarray(
        model.cf_generation(
            label_real=label_real_test,
            label_cf=label_cf_test,
            x=x_test,
            y=y_factual_test,
        )
    )
    m = counterfactual_mae_mbe(y_counterfactual_test, pred)
    print(f"[{name}] CF MAE={m['cf_mae']:.6f} | CF MBE={m['cf_mbe']:.6f}")

    cf_from_y = lambda y: np.asarray(
        model.cf_generation(
            label_real=label_real_test,
            label_cf=label_cf_test,
            x=x_test,
            y=y,
        )
    )
    mv = added_variations_relative(
        cf_from_y,
        y_factual=y_factual_test,
        seq_length=HORIZON,
        ini_start=2,
        num_windows=3,
        window_len=4,
    )
    print(
        f"[{name}] Added variations: total={mv['total_rel']:.6f} | "
        f"altered={mv['altered_steps_rel']:.6f} | unaltered={mv['unaltered_steps_rel']:.6f}"
    )

    ma = axiomatic_metrics(
        model,
        predictor,
        label_real=label_real_test,
        label_cf=label_cf_test,
        x=x_test,
        y_factual=y_factual_test,
    )
    print(
        f"[{name}] Axiomatic: composition={ma['composition']:.6f} | "
        f"reversibility={ma['reversibility']:.6f} | effectiveness={ma['effectiveness']:.6f}"
    )


## CEPAE

In [ ]:
model_cepae = CEPAE(
    seq_len=HORIZON,
    latent_dim=7,
    feat_dim=1,
    hidden_layer_sizes=[100, 200],
    Lambda=0.19,
)
model_cepae.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=model_cepae.loss_,
    metrics=[model_cepae.reconstruction, model_cepae.regularization],
)
model_cepae.fit(
    [train_labels, x_train, y_train],
    y_train,
    epochs=350,
    batch_size=32,
    validation_data=([eval_labels, x_eval, y_eval], y_eval),
    verbose=0,
)
evaluate_counterfactuals(model_cepae, "CEPAE")


## CVAE

In [ ]:
model_cvae = CVAE(
    seq_len=HORIZON,
    latent_dim=3,
    feat_dim=1,
    hidden_layer_sizes=[100, 200],
    recon_weight=200,
)
model_cvae.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=model_cvae.loss_,
    metrics=[model_cvae.reconstruction, model_cvae.kl],
)
model_cvae.fit(
    [train_labels, x_train, y_train],
    y_train,
    epochs=350,
    batch_size=32,
    validation_data=([eval_labels, x_eval, y_eval], y_eval),
    verbose=0,
)
evaluate_counterfactuals(model_cvae, "CVAE")


## CAAE

In [ ]:
model_caae = CAAE(
    seq_len=HORIZON,
    latent_dim=7,
    feat_dim=1,
    hidden_layer_sizes=[100, 200],
)

BATCH_SIZE = 32
train_ds = (
    tf.data.Dataset.from_tensor_slices((train_labels, x_train, y_train))
    .shuffle(buffer_size=len(x_train), seed=SEED)
    .batch(BATCH_SIZE)
)
eval_ds = tf.data.Dataset.from_tensor_slices((eval_labels, x_eval, y_eval)).batch(BATCH_SIZE)

max_iterations = int(np.ceil(len(x_train) / BATCH_SIZE) * 350)

train_caae(
    model_caae,
    train_ds,
    eval_ds,
    epochs=350,
    optimizer=Adam(learning_rate=1e-4),
    max_iterations=max_iterations,
    max_lambda=8.9,
    perform_evaluation=True,
)

evaluate_counterfactuals(model_caae, "CAAE")


## Forecast baseline

In [ ]:
# Forecast baseline: predicts y from (event, history).
y_train_2d = y_train[:, :, 0]
y_eval_2d = y_eval[:, :, 0]

model_forecast = ForecastModel(pred_steps=HORIZON)
model_forecast.compile(optimizer=Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
model_forecast.fit(
    [train_labels, x_train],
    y_train_2d,
    epochs=500,
    batch_size=32,
    validation_data=([eval_labels, x_eval], y_eval_2d),
    verbose=0,
)

pred_cf = np.asarray(model_forecast([label_cf_test, x_test]))[:, :, None]
m = counterfactual_mae_mbe(y_counterfactual_test, pred_cf)
print(f"[Forecast] CF MAE={m['cf_mae']:.6f} | CF MBE={m['cf_mbe']:.6f}")



## Adversarially balanced forecast (AB-LSTM)

In [ ]:
# Adversarially balanced forecast (AB-LSTM)
y_train_2d = y_train[:, :, 0]
y_eval_2d = y_eval[:, :, 0]

model_ab = AdversarialForecastModel(pred_steps=HORIZON)
trainer = Trainer(model=model_ab, optimizer=Adam(learning_rate=1e-3), max_steps=5000, max_lambda=2.0)

train_ds = tf.data.Dataset.from_tensor_slices((train_labels, x_train, y_train_2d)).shuffle(
    buffer_size=len(x_train), seed=SEED
).batch(32)
val_ds = tf.data.Dataset.from_tensor_slices((eval_labels, x_eval, y_eval_2d)).batch(32)

trainer.fit(train_ds, val_ds, epochs=200)

pred_cf = np.asarray(model_ab((label_cf_test, x_test), training=False))[:, :, None]
m = counterfactual_mae_mbe(y_counterfactual_test, pred_cf)
print(f"[AB-LSTM] CF MAE={m['cf_mae']:.6f} | CF MBE={m['cf_mbe']:.6f}")
